## Item 14.1 — "Destination": a flat JSON session log lets several already-finished `(query, ComposerAnswer)` bites persist under one fixed objective, so `synthesize` has something to read — without `run_query`, `composer`, or `reflect` ever needing to know a session exists.
*Budget: ~2 landed units expected.*

```mermaid
flowchart LR
    subgraph NOW["today"]
        RQ0["run_query: returns ComposerAnswer,\nprinted or discarded, nothing persists across calls"]
    end
    subgraph DEST["destination"]
        SB["save_bite(path, query, answer):\nappends to {objective, bites: [...]}"]
        LB["load_bites(path):\nreads the file back as objective + bites list"]
        SB --> LB
    end
    NOW ==> DEST
```

In [ ]:
# Landed 1 — ReflectionState resets fresh every run_query call with no field meant to outlive one query; the only thing worth persisting is the already-finished (query, ComposerAnswer) pair, saved to one flat JSON file per series ({"objective": ..., "bites": [...]}), never touching ReflectionState, composer, or reflect.

# Landed 2 — save_bite/load_bites take a Path (same DATA_DIR convention as catalog.py); the only disk-facing decision is Path.exists() (new series vs. existing one) — everything else is Pydantic's own model_dump()/model_validate() round-tripping ComposerAnswer through JSON.

## Item 14.2 — "Destination": `synthesize(objective, bites)` weaves several already-grounded, quote-backed `ComposerAnswer`s into one connective narrative relative to the objective, and names one uncovered angle plus a candidate next query — grounded only in what the bites themselves already established, never licensed to assert a connection they don't support, and never deciding on its own to run that candidate query.
*Budget: ~3 landed units expected.*

```mermaid
flowchart LR
    subgraph NOW["today"]
        S0["no synthesis step exists;\nbites sit logged, unused beyond that"]
    end
    subgraph DEST["destination"]
        S1["synthesize(objective, bites)\n-> Synthesis{narrative, uncovered_angle, candidate_query}\ngrounded only in the bites given"]
    end
    NOW ==> DEST
```

In [ ]:
# Landed 1 — every ComposerAnswer synthesize receives is already quote-backed (item 8), but a sentence connecting two bites is a new claim invented by this call, not grounded just because its ingredients were — the same fabrication risk the composer-injection idea was rejected over, one level removed.

# Landed 2 — the guard decomposes narrative into claims: list[SynthesisClaim], each naming which bite(s) it is based_on (mirroring unique_findings' per-entry attribution pattern); uncovered_angle/candidate_query stay a lighter prompt-level guard for now, not their own attribution field, matching the manual-first/don't-over-build-before-the-trial stance already applied to the CLI.

# Landed 3 — synthesize's body: section 1 renders each bite via the existing format_answer, tagged by its own query; section 2's prompt carries the grounding rule near-verbatim (claims must name based_on bites and never assert beyond them); section 3 is a single with_structured_output(Synthesis) call, no merge logic needed.

## Mapping the conceptual vocabulary (Theme / Objective / Query) onto `synthesis.py`'s actual fields, to guide docstring unification.

```mermaid
flowchart TB
    subgraph CONCEPT["conceptual (Claude web) — broad to narrow"]
        THEME["Theme\n(informal, never computed)"]
        OBJ["Objective\n(one bounded sentence, the north star)"]
        QRY["Query\n(one targeted, single-themed ask)"]
        THEME --> OBJ --> QRY
    end

    subgraph CODE["computational (synthesis.py)"]
        OBJP["objective: str\nsynthesize's fixed input"]
        BITEQ["bite.query: str\neach already-run Query\n(via save_bite/load_bites)"]
        CLAIMS["Synthesis.claims: list[SynthesisClaim]\nnarrative about the Objective"]
        CLAIM["SynthesisClaim.claim: str\none connective statement"]
        BASEDON["SynthesisClaim.based_on: list[str]\nwhich bite queries back this claim"]
        UNCOV["Synthesis.uncovered_angle: str | None\nfacet of Objective untouched"]
        CAND["Synthesis.candidate_query: str | None\na new, not-yet-run Query"]
    end

    OBJ -. "typed directly as" .-> OBJP
    QRY -. "typed directly as" .-> BITEQ

    OBJP --> CLAIMS
    OBJP --> UNCOV
    BITEQ --> CLAIMS
    BITEQ --> UNCOV
    CLAIMS --> CLAIM
    CLAIM --> BASEDON
    BASEDON -. "cites back" .-> BITEQ
    UNCOV --> CAND
    CAND -. "is itself a new" .-> QRY
```

## Updated mapping after the Objective -> Thesis rename, and after adding `facet`/`note` to each bite: same conceptual/computational structure as before, now showing `facet` as a real input to `synthesize` and `note` as a genuine dead end (saved, never read back).

```mermaid
flowchart TB
    subgraph CONCEPT["conceptual (Claude web) — broad to narrow"]
        THEME["Theme\n(informal, never computed)"]
        THESIS_C["Thesis\n(one bounded, declarative sentence)"]
        QRY["Query\n(one targeted, single-themed ask)"]
        FACET_C["facet\n(which part of the Thesis this query targets)"]
        NOTE_C["doubt / note\n(Claude web's point-5 risk flag, free text)"]
        THEME --> THESIS_C --> QRY
        QRY -.-> FACET_C
        QRY -.-> NOTE_C
    end

    subgraph CODE["computational (synthesis.py)"]
        THESISP["thesis: str\nsynthesize's fixed input"]
        BITEQ["bite.query: str\neach already-run Query"]
        FACETP["bite.facet: str | None\nrendered into synthesize's prompt"]
        NOTEP["bite.note: str | None\nsaved by save_bite,\nnever returned by load_bites — a true dead end"]
        CLAIMS["Synthesis.claims: list[SynthesisClaim]\nnarrative about the Thesis"]
        CLAIM["SynthesisClaim.claim: str\none connective statement"]
        BASEDON["SynthesisClaim.based_on: list[str]\nwhich bite queries back this claim"]
        UNCOV["Synthesis.uncovered_angle: str | None\nfacet of Thesis untouched"]
        CAND["Synthesis.candidate_query: str | None\na new, not-yet-run Query"]
    end

    THESIS_C -. "typed directly as" .-> THESISP
    QRY -. "typed directly as" .-> BITEQ
    FACET_C -. "typed directly as" .-> FACETP
    NOTE_C -. "typed directly as" .-> NOTEP

    THESISP --> CLAIMS
    THESISP --> UNCOV
    BITEQ --> CLAIMS
    BITEQ --> UNCOV
    FACETP --> CLAIMS
    FACETP --> UNCOV
    CLAIMS --> CLAIM
    CLAIM --> BASEDON
    BASEDON -. "cites back" .-> BITEQ
    UNCOV --> CAND
    CAND -. "is itself a new" .-> QRY
```

## `facet` deep-dive — benefit, constraint, and vocab-mismatch interaction

In [ ]:
# Q1 - the benefit facet brings in ?
# A1 - facet carries Claude web's query-design intent -- which axis of the thesis a query was built to test -- into synthesize, something the finished query+answer alone wouldn't reliably reveal.

# Q2 - the benefit that a constrained label matched mechanically would bring to facet / the system as a whole ?
# A2 - The "label matched" is to check if the wording used in facet is also used on the thesis sentence itself. It is a check for vocab sharing between thesis and facet.  A constrained, enum-like label (matched against the thesis's own named axes) would make
#      coverage-checking a reliable lookup instead of a fuzzy text comparison, and would catch a mislabeled or inconsistent facet at the schema boundary instead of letting it silently drift.

# Q3 - how vocab mismatch affects facet's capability.
# A3 - vocabulary mismatch can make facet's claim hollow: the label still says "origin" even if that bite's retrieval actually came back thin or silent, so synthesize can see an axis as "covered" when the real evidence behind it quietly failed.

## Verdict: `facet`/`note` stripped — neither survived scrutiny of what Claude web uniquely contributes. Final conceptual/computational mapping, two carriers only.

```mermaid
flowchart TB
    subgraph CONCEPT["conceptual (Claude web) — broad to narrow"]
        THEME["Theme\n(informal, never computed)"]
        THESIS_C["Thesis\n(one bounded, declarative sentence)"]
        QRY["Query\n(one targeted, single-themed ask)"]
        VAR["variants\n(2-3 per-book-register alternatives,\nnever typed in — human's own fallback menu)"]
        THEME --> THESIS_C --> QRY
        QRY -.-> VAR
    end

    subgraph CODE["computational (synthesis.py)"]
        THESISP["thesis: str\nsynthesize's fixed input"]
        BITEQ["bite.query: str\neach already-run Query"]
        CLAIMS["Synthesis.claims: list[SynthesisClaim]\nnarrative about the Thesis"]
        CLAIM["SynthesisClaim.claim: str\none connective statement"]
        BASEDON["SynthesisClaim.based_on: list[str]\nwhich bite queries back this claim"]
        UNCOV["Synthesis.uncovered_angle: str | None\npart of Thesis untouched"]
        CAND["Synthesis.candidate_query: str | None\na new, not-yet-run Query"]
    end

    THESIS_C -. "typed directly as" .-> THESISP
    QRY -. "typed directly as" .-> BITEQ

    THESISP --> CLAIMS
    THESISP --> UNCOV
    BITEQ --> CLAIMS
    BITEQ --> UNCOV
    CLAIMS --> CLAIM
    CLAIM --> BASEDON
    BASEDON -. "cites back" .-> BITEQ
    UNCOV --> CAND
    CAND -. "is itself a new" .-> QRY
```

Why `variants` has no computational node: it's never typed into ParallelProse at all — it's a
human-facing fallback kept uncontaminated by any failed retrieval attempt, unlike the system's own
`narrowed_query` retry (inside `agent.py`, not this diagram's scope).